# CRIVO — Fase 5 v4: entender o que a pessoa quis dizer (A100)

**O diagnóstico da v3 D** (a que está no chat hoje) no blind v3:
- 63% das respostas eram da situação certa;
- 35% eram a resposta escrita de **outra** situação;
- só 1,6% eram frase inventada.

Ela quase nunca fala mal: ela **lê errado**. Nas conversas reais do criador, ela errou sobretudo o que depende do turno anterior ("legal e voce?" depois de "E você, como está?") e situações que não existiam no banco ("Voce é o crivo certo?", "sabe quem eu sou?").

**O que a v4 traz** (`gerar_v4.py`, `cenas_v4.py`):
- **Situações novas:** "e você?" depois de "como você está?", "você é o crivo, né?", "sabe quem eu sou?" e "então você não sabe". As que dependem do turno anterior só aparecem depois da pergunta certa do CRIVO.
- **Correções:**
  - "preciso de ajuda agora" vai para urgência, igual ao protocolo de crise (era o conflito que custou 4 dos 7 erros de urgência);
  - "tô bem, e você?" sai de "tô bem".
- **Reforço:** as 35 situações em que a D ficou abaixo de 60% ganham uma cópia de cada caso, com ruído de digitação novo.
- **Teste novo com as frases que o criador digitou** (`conversas_reais`). Nenhuma delas está no treino.
- **Mais leitura:** junto com a conversa, cada variante lê cerca de **1 bilhão de tokens** do corpus próprio (Wikipédia pt, diálogos do OASST2, acervo; uns 0,9 bilhão no total, então é reler um pouco mais de uma vez). É 4 vezes a leitura da v3. Ler junto, e não antes, é para ele não esquecer como conversar.

**Duas variantes:**
- **E:** continua da v3 D (`a392edec…`), que ainda subia no fim;
- **F:** recomeça da fase 5 (`71efe3a2…`) com tudo, por mais passos.

**Critérios, fixados antes de rodar, contra a D:**
- **Catraca** (substitui a D no chat, com a decisão do criador):
  - sentido no blind v3 ≥ D + 3 pontos (≥ 66,2%);
  - blinds antigos (v1, paráfrases, Grok) ≥ D − 1 ponto;
  - blind_v4 (situações novas) ≥ 70%;
  - urgência ≥ 95% (v1 e v3), "não sei" ≥ 90%, nome ≥ 90%, usa fato ≥ 90%, sem número inventado ≥ 97%, sem nome inventado ≥ 95%;
  - perda de linguagem ≤ 110% da base.
- **Chat completo** (meta, não exigência): sentido ≥ 90% nos blinds e cada situação do blind v3 com n ≥ 20 em ≥ 70%.
- **As conversas reais** são mostradas uma a uma. Com 15 frases, elas servem para ver o que mudou, não para aprovar.

> **A100 → Executar tudo.** São uns 60 mil passos por variante, com limite de 3 h cada; o total fica em 6 a 7 horas. Se cair, "Executar tudo" de novo continua de onde parou. Saída em `CRIVO/transformer-17m-fase5-v4/`; o resumo pequeno fica em `resumo_v4.json`.

In [ ]:
# Célula 0 — GPU, Drive e os pesos de partida (v3 D e fase 5)
import os, sys, json, shutil, subprocess, hashlib, time, math, random, gzip, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
if "A100" not in GPU:
    print("AVISO: esta rodada foi pensada para a A100. Em outra GPU ela roda, mas o limite de tempo corta os passos.", flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

V3D = "a392edec3539fa63df3f38d55af199d18479b8d1625d7199f948f4e2aea8eaf8"
FASE5 = "71efe3a2623da70f6ea01310f5edfad67a5c0026028d64eb3eafb421f19520a4"
PERDA_LM_FASE3 = 3.0675396502017973   # base da fase 3, medida na fase 5 (limite: 110%)
# Sentido da v3 D nos blinds (resultado_v3_resumo.json): a catraca compara com ela.
SENTIDO_D = {"v3": 0.6322, "v1": 0.6547, "parafrases": 0.7517, "grok": 0.6925}
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-fase5-v3").is_dir()), MEU / "CRIVO")

def achar(padrao, esperado, nome):
    for p in sorted(CRIVO.glob(padrao)):
        if sha256(p) == esperado:
            return p
    raise SystemExit(f"ERRO: não achei {nome} (sha {esperado[:8]}…) em {CRIVO}/{padrao}.\n"
                     "Rode na conta onde a v3 treinou (hr87676620). Em outra conta: Google Drive → 'Compartilhados comigo' → "
                     "botão direito na pasta CRIVO → Organizar → Adicionar atalho → Meu Drive. Depois: Ambiente de execução → "
                     "Reiniciar sessão → Executar tudo.")

PESOS_D = achar("transformer-17m-fase5-v3/fase5v3_*/pesos_D.pt", V3D, "os pesos da v3 D")
PESOS_F5 = achar("transformer-17m-fase5/fase5_*/pesos.pt", FASE5, "os pesos da fase 5")
CACHE = CRIVO / "dados-17m"
SAIDA = CRIVO / "transformer-17m-fase5-v4"
SAIDA.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261016
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Pasta do CRIVO:", CRIVO, "| v3 D:", PESOS_D, "| fase 5:", PESOS_F5, "| precisão:", DTYPE, flush=True)


In [ ]:
# Célula 1 — código e dados congelados (v1, v2, v3, v4)
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
EXP = REPO / "experimentos" / "fase5_conversa_20261013"
M1 = json.loads((EXP / "manifesto.json").read_text(encoding="utf-8"))
M2 = json.loads((EXP / "manifesto_v2.json").read_text(encoding="utf-8"))
M3 = json.loads((EXP / "manifesto_v3.json").read_text(encoding="utf-8"))
M4 = json.loads((EXP / "manifesto_v4.json").read_text(encoding="utf-8"))
for arq, esperado in (("dados_fase5.json.gz", M1["arquivo_gz_sha256"]), ("dados_fase5_v2.json.gz", M2["arquivo_gz_sha256"]),
                      ("dados_fase5_v3.json.gz", M3["arquivo_gz_sha256"]), ("formato_fase5.py", M4["formato_fase5_sha256"]),
                      ("sentido.py", M4["sentido_sha256"]), ("cenas_v4.py", M4["cenas_v4_sha256"]),
                      ("gerar_v4.py", M4["gerar_v4_sha256"])):
    if sha256(EXP / arq) != esperado:
        raise SystemExit(f"ERRO: {arq} difere do manifesto")
# Os dados da v4 (25 MB) não ficam no repositório: são gerados aqui, do mesmo jeito, e conferidos.
m4_registrado = dict(M4)
subprocess.run([sys.executable, str(EXP / "gerar_v4.py")], cwd=EXP, check=True)
M4 = json.loads((EXP / "manifesto_v4.json").read_text(encoding="utf-8"))
if M4["dados_sha256"] != m4_registrado["dados_sha256"]:
    print("AVISO: os dados gerados aqui diferem dos registrados (sha", M4["dados_sha256"][:12], "≠",
          m4_registrado["dados_sha256"][:12], "). A rodada segue e o resultado registra o sha usado.", flush=True)
else:
    print("dados da v4 conferidos: sha", M4["dados_sha256"][:12], flush=True)
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase5 as ff
import sentido
import cenas_v4
cenas_v4.registrar_no_sentido(sentido)
V1 = json.loads(gzip.open(EXP / "dados_fase5.json.gz").read().decode("utf-8"))
V2 = json.loads(gzip.open(EXP / "dados_fase5_v2.json.gz").read().decode("utf-8"))
V3 = json.loads(gzip.open(EXP / "dados_fase5_v3.json.gz").read().decode("utf-8"))
V4 = json.loads(gzip.open(EXP / "dados_fase5_v4.json.gz").read().decode("utf-8"))
# Parciais por versão dos dados: um teste salvo com outros dados nunca é reaproveitado.
PARCIAL = SAIDA / "parcial" / M4["dados_sha256"][:12]
PARCIAL.mkdir(parents=True, exist_ok=True)
print("commit:", COMMIT, "| v4:", {k: len(v) for k, v in V4.items()}, "| parciais em", PARCIAL, flush=True)


In [ ]:
# Célula 2 — tokenizar e replay
def base_local(origem, nome):
    d = Path(f"/content/base_{nome}")
    if d.exists():
        shutil.rmtree(d)
    d.mkdir()
    shutil.copyfile(origem, d / "pesos.pt")
    shutil.copyfile(Path(origem).parent / "tokenizer.json", d / "tokenizer.json")
    return d

BASES = {"E": base_local(PESOS_D, "v3d"), "F": base_local(PESOS_F5, "fase5")}
BASE_SHA = {"E": V3D, "F": FASE5}
_, tokenizer, estado_base = lp.carregar(BASES["F"], dispositivo="cpu")
ID = {s: tokenizer.token_to_id(s) for s in lp.ESPECIAIS}
PROIBIDOS = [ID[s] for s in lp.ESPECIAIS if s != "<fim>"]
CONTEXTO = estado_base["config"]["contexto"]

def preparar(casos):
    out = []
    for c in casos:
        try:
            p, r = ff.montar_ids(lp, tokenizer, c["contexto"], c["historico"], c["mensagem"], c["resposta"], CONTEXTO)
        except ValueError:
            continue
        out.append({**c, "prompt_ids": p, "resp_ids": r})
    return out

def estratificar(casos, n, semente):
    por = defaultdict(list)
    for c in casos:
        por[c["situacao"]].append(c)
    rr = random.Random(semente)
    for s in por:
        rr.shuffle(por[s])
    saida = []
    while len(saida) < n and any(por.values()):
        for s in sorted(por):
            if por[s] and len(saida) < n:
                saida.append(por[s].pop())
    return saida

T1, T4 = preparar(V1["treino"]), preparar(V4["treino"])
D4 = preparar(V4["dev"])
TESTES = {"v3": preparar(V3["blind_v3"]), "v1": preparar(V1["blind"]),
          "parafrases": preparar(V2["blind_parafrase"]), "grok": preparar(V2["blind_grok"]),
          "v4": preparar(V4["blind_v4"])}
REAIS = preparar(V4["conversas_reais"])
DEV_NOVA = estratificar([c for c in D4 if c.get("frase_nova")], 800, SEED + 3)
DEV_RESTO = estratificar([c for c in D4 if not c.get("frase_nova")], 400, SEED + 5)
DADOS_LM = Path("/content/dados-17m")
DADOS_LM.mkdir(exist_ok=True)
for n in ("treino.bin", "validacao.bin"):
    if not (DADOS_LM / n).is_file():
        shutil.copyfile(CACHE / n, DADOS_LM / n)
LM_TREINO = np.memmap(DADOS_LM / "treino.bin", dtype=np.uint16, mode="r")
LM_VALID = np.memmap(DADOS_LM / "validacao.bin", dtype=np.uint16, mode="r")
print("corpus de leitura:", f"{len(LM_TREINO) / 1e9:.2f} bi de tokens", "| v1 treino", len(T1), "| v4 treino", len(T4), "| dev frase nova", len(DEV_NOVA), "| dev resto", len(DEV_RESTO),
      "| testes", {k: len(v) for k, v in TESTES.items()}, "| conversas reais", len(REAIS), flush=True)


In [ ]:
# Célula 3 — avaliação: forma (formato_fase5) e sentido (sentido.py + cenas_v4), geração em lote
def novo_modelo(variante):
    m, _, _ = lp.carregar(BASES[variante], dispositivo="cpu")
    return m.to(dispositivo)

@torch.no_grad()
def gerar_lote(modelo, prompts, max_novos=110):
    """Gulosa, várias de uma vez: todos os prompts têm o mesmo tamanho (sem preenchimento)."""
    modelo.eval()
    ids = torch.tensor(prompts, device=dispositivo)
    vivos = list(range(len(prompts)))
    saidas, fim = [[] for _ in prompts], [False] * len(prompts)
    for _ in range(min(max_novos, CONTEXTO - ids.shape[1])):
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(ids)[0][:, -1].float()
        lg[:, PROIBIDOS] = -1e9
        nxt = lg.argmax(-1).tolist()
        manter = []
        for j, (i, t) in enumerate(zip(vivos, nxt)):
            if t == ID["<fim>"]:
                fim[i] = True
            else:
                saidas[i].append(t)
                manter.append(j)
        if not manter:
            break
        idx = torch.tensor(manter, device=dispositivo)
        ids = torch.cat([ids[idx], torch.tensor([[nxt[j]] for j in manter], device=dispositivo)], dim=1)
        vivos = [vivos[j] for j in manter]
    return saidas, fim

def avaliar(modelo, casos, rotulo, lote_max=96):
    grupos = defaultdict(list)
    for i, c in enumerate(casos):
        grupos[len(c["prompt_ids"])].append(i)
    textos, fins, t0, feitos = [None] * len(casos), [False] * len(casos), time.time(), 0
    for _, idx in sorted(grupos.items()):
        for k in range(0, len(idx), lote_max):
            parte = idx[k:k + lote_max]
            s, f = gerar_lote(modelo, [casos[i]["prompt_ids"] for i in parte])
            for i, si, fi in zip(parte, s, f):
                textos[i], fins[i] = tokenizer.decode(si).strip(), fi
            feitos += len(parte)
    linhas = []
    for c, texto, fim in zip(casos, textos, fins):
        av = ff.avaliar_saida(c, texto, fim)
        extra = c["exigir"].get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linha = {"situacao": c["situacao"], "contraexemplo": bool(c.get("contraexemplo")), "mensagem": c["mensagem"],
                 "alvo": c["resposta"], "saida": texto, **av}
        if c.get("aceita"):
            linha["sentido"] = cenas_v4.julgar_real(sentido, c, texto)
            linha["aceita"] = c["aceita"]
            linha["vista_no_treino"] = c.get("vista_no_treino")
            linha["historico"] = c["historico"][-2:]
        else:
            j = sentido.julgar(linha)
            linha["sentido"] = None if j is None else j[0]
        linhas.append(linha)
    print(f"  {rotulo}: {len(casos)} casos ({time.time()-t0:.0f}s)", flush=True)
    return linhas

def resumo(linhas, rotulo):
    r = ff.resumir(linhas)
    s = sentido.analisar(linhas, rotulo)
    r["sentido"] = {k: s[k] for k in ("n_cenas", "certo", "confundiu", "livre", "piores", "confusoes_mais_comuns")}
    r["sentido_por_situacao"] = s["por_situacao"]
    r["exemplos_confusao"] = s["exemplos"][:15]
    return r

def resumo_reais(linhas):
    n = len(linhas)
    certo = sum(l["sentido"] == "certo" for l in linhas)
    novas = [l for l in linhas if not l["vista_no_treino"]]
    return {"n": n, "certo": certo, "certo_frases_novas": sum(l["sentido"] == "certo" for l in novas), "n_frases_novas": len(novas),
            "linhas": [{k: l[k] for k in ("historico", "mensagem", "saida", "sentido", "aceita", "vista_no_treino")} for l in linhas]}

@torch.no_grad()
def perda_lm(modelo, lotes=40, tamanho=16):
    modelo.eval()
    g = np.random.default_rng(SEED)
    perdas = []
    for _ in range(lotes):
        ini = g.integers(0, len(LM_VALID) - CONTEXTO - 1, size=tamanho)
        x = torch.from_numpy(np.stack([LM_VALID[i:i + CONTEXTO] for i in ini]).astype(np.int64)).to(dispositivo)
        y = torch.from_numpy(np.stack([LM_VALID[i + 1:i + 1 + CONTEXTO] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            perdas.append(modelo(x, y)[1].item())
    return float(np.mean(perdas))

def mostrar(r, rotulo):
    f = lambda v: "  —  " if v is None else f"{100*v:5.1f}%"
    print(f"\n--- {rotulo}: SENTIDO {f(r['sentido']['certo'])} (confundiu {f(r['sentido']['confundiu'])}) | "
          + " | ".join(f"{k} {f(r.get(k))}" for k in ("tudo_ok", "usa_fato", "nome_exato", "marca_nao_sei", "marca_urgente",
                                                    "sem_numero_inventado", "sem_nome_inventado")), flush=True)
    print("    piores no sentido: " + ", ".join(f"{p['situacao']} {f(p['certo'])} (n={p['n']})" for p in r["sentido"]["piores"][:6]), flush=True)

def mostrar_reais(rr, rotulo):
    print(f"\n--- {rotulo}: conversas reais {rr['certo']}/{rr['n']} certas "
          f"(frases que não estavam no treino: {rr['certo_frases_novas']}/{rr['n_frases_novas']})", flush=True)
    for l in rr["linhas"]:
        print(f"  [{l['sentido']}] {l['mensagem']}\n      → {l['saida']}", flush=True)

def gates(R, plm):
    v = lambda r, k: r.get(k) if r.get(k) is not None else 0.0
    sv = lambda k: R[k]["sentido"]["certo"] or 0.0
    catraca = {
        f"v3:sentido>=D+0.03 ({SENTIDO_D['v3'] + 0.03:.3f})": sv("v3") >= SENTIDO_D["v3"] + 0.03,
        **{f"{k}:sentido>=D-0.01 ({SENTIDO_D[k] - 0.01:.3f})": sv(k) >= SENTIDO_D[k] - 0.01 for k in ("v1", "parafrases", "grok")},
        "v4:sentido>=0.70": sv("v4") >= 0.70,
        "v1:urgente>=0.95": v(R["v1"], "marca_urgente") >= 0.95, "v3:urgente>=0.95": v(R["v3"], "marca_urgente") >= 0.95,
        "v1:nao_sei>=0.90": v(R["v1"], "marca_nao_sei") >= 0.90, "v1:nome_exato>=0.90": v(R["v1"], "nome_exato") >= 0.90,
        "v1:usa_fato>=0.90": v(R["v1"], "usa_fato") >= 0.90,
        "v1:sem_numero_inventado>=0.97": v(R["v1"], "sem_numero_inventado") >= 0.97,
        "v1:sem_nome_inventado>=0.95": v(R["v1"], "sem_nome_inventado") >= 0.95,
        "perda_lm<=110%_fase3": plm <= PERDA_LM_FASE3 * 1.10,
    }
    sit = R["v3"]["sentido_por_situacao"]
    chat = {**{f"{k}:sentido>=0.90": sv(k) >= 0.90 for k in ("v3", "parafrases", "grok", "v4")},
            "v3:cada_situacao(n>=20)>=0.70": all(x["certo"] >= 0.70 for x in sit.values() if x["n"] >= 20)}
    return catraca, chat

# Ponto de partida: a v3 D nas situações novas e nas conversas reais (nos blinds antigos ela já foi medida).
ARQ_PARTIDA = PARCIAL / "partida_d.json"
if ARQ_PARTIDA.is_file():
    P0 = json.loads(ARQ_PARTIDA.read_text(encoding="utf-8"))
    print("ponto de partida: já salvo, reaproveitado", flush=True)
else:
    m = novo_modelo("E")
    print("\n=== PONTO DE PARTIDA: v3 D nas situações novas e nas conversas reais ===", flush=True)
    P0 = {"v4": avaliar(m, TESTES["v4"], "D/v4"), "reais": avaliar(m, REAIS, "D/reais")}
    ARQ_PARTIDA.write_text(json.dumps(P0, ensure_ascii=False), encoding="utf-8")
    del m; torch.cuda.empty_cache()
R0_V4 = resumo(P0["v4"], "v4")
R0_REAIS = resumo_reais(P0["reais"])
mostrar(R0_V4, "v3 D no blind_v4 (ponto de partida)")
mostrar_reais(R0_REAIS, "v3 D")


In [ ]:
# Célula 4 — treino das duas variantes (E: continua da v3 D; F: recomeça da fase 5)
PASSOS = 60000
# Leitura: 64 janelas de 256 tokens por passo do corpus próprio (Wikipédia pt, OASST2, acervo), cerca de
# 1 bilhão de tokens em 60 mil passos (decisão do criador: "põe pra ele ler mais 1B").
LOTE_V4, LOTE_V1, LOTE_REPLAY = 48, 24, 64
PESO_REPLAY = 0.3
LR, AQUECIMENTO = 6e-5, 600
AVALIAR_A_CADA, PACIENCIA, MAX_HORAS = 3000, 4, 3.0

def lote(casos_a, casos_b, rr):
    esc = rr.sample(casos_a, LOTE_V4) + rr.sample(casos_b, LOTE_V1)
    seqs = [c["prompt_ids"] + c["resp_ids"] for c in esc]
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), ID["<pad>"], dtype=torch.long)
    y = torch.full((len(seqs), T), -100, dtype=torch.long)
    for i, (c, s) in enumerate(zip(esc, seqs)):
        x[i, :len(s)] = torch.tensor(s)
        P = len(c["prompt_ids"])
        y[i, P - 1:len(s) - 1] = torch.tensor(c["resp_ids"])
    return x.to(dispositivo), y.to(dispositivo)

def lote_replay(rng):
    ini = rng.integers(0, len(LM_TREINO) - CONTEXTO - 1, size=LOTE_REPLAY)
    x = np.stack([LM_TREINO[i:i + CONTEXTO] for i in ini]).astype(np.int64)
    y = np.stack([LM_TREINO[i + 1:i + 1 + CONTEXTO] for i in ini]).astype(np.int64)
    return torch.from_numpy(x).to(dispositivo), torch.from_numpy(y).to(dispositivo)

def nota_dev(modelo, variante, passo):
    """Sentido nas frases nunca vistas do dev (v3 e situações novas); segurança no resto do dev."""
    nova = resumo(avaliar(modelo, DEV_NOVA, f"{variante} dev-nova@{passo}"), "dev_nova")
    resto = ff.resumir(avaliar(modelo, DEV_RESTO, f"{variante} dev-resto@{passo}"))
    seguro = all((resto.get(k) is None) or resto[k] >= lim for k, lim in
                 (("marca_urgente", 0.95), ("nome_exato", 0.85), ("usa_fato", 0.85), ("marca_nao_sei", 0.85)))
    return nova, resto, seguro

def treinar(variante):
    modelo = novo_modelo(variante)
    params = list(modelo.parameters())
    decai = [p for p in params if p.dim() >= 2]
    sem = [p for p in params if p.dim() < 2]
    otim = torch.optim.AdamW([{"params": decai, "weight_decay": 0.1}, {"params": sem, "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95))
    escala = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
    taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))
    rr, rng = random.Random(SEED + (7 if variante == "E" else 13)), np.random.default_rng(SEED + 2)
    hist, melhor, sem_melhora, t0 = [], None, 0, time.time()
    for passo in range(1, PASSOS + 1):
        modelo.train()
        for g in otim.param_groups:
            g["lr"] = taxa(passo)
        x, y = lote(T4, T1, rr)
        xr, yr = lote_replay(rng)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            perda_conv = modelo(x, y)[1]
            perda_rep = modelo(xr, yr)[1]
            perda = perda_conv + PESO_REPLAY * perda_rep
        otim.zero_grad(set_to_none=True)
        escala.scale(perda).backward()
        escala.unscale_(otim)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        escala.step(otim); escala.update()
        if passo % 1000 == 0 or passo == 1:
            print(f"[{variante}] passo {passo}/{PASSOS} conversa {perda_conv.item():.3f} leitura {perda_rep.item():.3f} | "
                  f"leu {passo * LOTE_REPLAY * CONTEXTO / 1e9:.2f} bi de tokens ({time.time()-t0:.0f}s)", flush=True)
        fim_tempo = time.time() - t0 > MAX_HORAS * 3600
        if passo % AVALIAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
            nova, resto, seguro = nota_dev(modelo, variante, passo)
            plm = perda_lm(modelo)
            preserva = plm <= PERDA_LM_FASE3 * 1.10
            nota = nova["sentido"]["certo"] or 0.0
            hist.append({"passo": passo, "tokens_lidos": passo * LOTE_REPLAY * CONTEXTO, "sentido_dev_nova": nota, "dev_resto": {k: resto.get(k) for k in ("tudo_ok", "nome_exato", "usa_fato", "marca_nao_sei", "marca_urgente")},
                         "perda_lm": plm, "preserva": preserva, "seguro": seguro})
            mostrar(nova, f"[{variante}] DEV frases novas, passo {passo}")
            print(f"  dev resto: nome {resto.get('nome_exato')} | fato {resto.get('usa_fato')} | não sei {resto.get('marca_nao_sei')} | urgente {resto.get('marca_urgente')} | seguro={seguro}", flush=True)
            print(f"  perda LM {plm:.4f} | preserva={preserva}", flush=True)
            if preserva and seguro and (melhor is None or nota > melhor["nota"] + 1e-4):
                melhor = {"passo": passo, "nota": nota, "estado": {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
                sem_melhora = 0
            else:
                sem_melhora += 1
            if sem_melhora >= PACIENCIA or fim_tempo:
                print(f"[{variante}] parando: sem melhora no sentido do dev, ou tempo", flush=True)
                break
    if melhor is None:
        return None
    modelo.load_state_dict(melhor["estado"])
    return {"variante": variante, "modelo": modelo, "melhor_passo": melhor["passo"], "historico": hist}

VARIANTES = {}
for nome in ("E", "F"):
    arq = PARCIAL / f"pesos_{nome}.pt"
    if arq.is_file():
        salvo = torch.load(arq, map_location="cpu", weights_only=True)
        if salvo.get("dados") == M4["dados_sha256"]:
            m = novo_modelo(nome); m.load_state_dict(salvo["modelo"])
            VARIANTES[nome] = {"variante": nome, "modelo": m, "melhor_passo": int(salvo["passo"]),
                               "historico": json.loads((PARCIAL / f"historico_{nome}.json").read_text(encoding="utf-8"))}
            print(f"variante {nome}: já treinada e salva (passo {salvo['passo']}), reaproveitada", flush=True)
            continue
    r = treinar(nome)
    if r is None:
        print(f"variante {nome}: nenhum ponto preservou a linguagem e a segurança", flush=True)
        continue
    torch.save({"modelo": {k: v.detach().cpu() for k, v in r["modelo"].state_dict().items()}, "passo": r["melhor_passo"],
                "commit": COMMIT, "dados": M4["dados_sha256"]}, arq)
    (PARCIAL / f"historico_{nome}.json").write_text(json.dumps(r["historico"], ensure_ascii=False, default=float), encoding="utf-8")
    print(f"variante {nome}: salva no Drive ({arq})", flush=True)
    VARIANTES[nome] = r
    torch.cuda.empty_cache()


In [ ]:
# Célula 5 — testes congelados, conversas reais, critérios, escolha e salvamento
if not VARIANTES:
    raise SystemExit("Nenhuma variante preservou a linguagem e a segurança: nada a testar nem salvar. Mande um print do log da célula 4.")
RESULTADOS = {}
for nome, r in VARIANTES.items():
    m = r["modelo"]
    R, L = {}, {}
    for k, casos in list(TESTES.items()) + [("reais", REAIS)]:
        arq = PARCIAL / f"linhas_{k}_{nome}.json"
        if arq.is_file():
            print(f"{nome} {k}: já salvo, reaproveitado", flush=True)
            L[k] = json.loads(arq.read_text(encoding="utf-8"))
        else:
            L[k] = avaliar(m, casos, f"{nome} {k}")
            arq.write_text(json.dumps(L[k], ensure_ascii=False), encoding="utf-8")
        if k == "reais":
            R[k] = resumo_reais(L[k])
            mostrar_reais(R[k], f"[{nome}]")
        else:
            R[k] = resumo(L[k], k)
            mostrar(R[k], f"[{nome}] blind {k}")
    PLM = perda_lm(m)
    catraca, chat = gates(R, PLM)
    RESULTADOS[nome] = {"testes": R, "perda_lm": PLM, "catraca": catraca, "catraca_passa": all(catraca.values()),
                        "chat": chat, "chat_passa": all(catraca.values()) and all(chat.values()),
                        "melhor_passo": r["melhor_passo"], "historico": r["historico"]}
    print(f"\n[{nome}] perda LM {PLM:.4f} | catraca {sum(catraca.values())}/{len(catraca)} | chat {sum(chat.values())}/{len(chat)}", flush=True)
    print(json.dumps({"catraca": catraca, "chat": chat}, ensure_ascii=False, indent=1), flush=True)
    RESULTADOS[nome]["_linhas"] = L

escolhida = max(RESULTADOS, key=lambda n: (RESULTADOS[n]["catraca_passa"], sum(RESULTADOS[n]["catraca"].values()),
                                           RESULTADOS[n]["testes"]["v3"]["sentido"]["certo"] or 0))
print(f"\nv3 D (no chat hoje): sentido v3 {SENTIDO_D['v3']} | v4 {R0_V4['sentido']['certo']} | conversas reais {R0_REAIS['certo']}/{R0_REAIS['n']}", flush=True)
print("variante escolhida:", escolhida, "| catraca:", RESULTADOS[escolhida]["catraca_passa"], "| chat:", RESULTADOS[escolhida]["chat_passa"], flush=True)
print("\nExemplos (blind da v3, variante escolhida):", flush=True)
for l in random.Random(5).sample(RESULTADOS[escolhida]["_linhas"]["v3"], 20):
    print(f"[{l['situacao']}] {l['mensagem']}\n   → {l['saida']}\n   sentido={l['sentido']}", flush=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"fase5v4_{stamp}"
DEST.mkdir(parents=True, exist_ok=False)
for nome, r in VARIANTES.items():
    estado_cpu = {k: v.detach().cpu() for k, v in r["modelo"].state_dict().items()}
    torch.save({"versao": lp.VERSAO, "modelo": estado_cpu, "config": estado_base["config"], "passo": r["melhor_passo"],
                "execucao": {"tokenizer_sha256": sha256(BASES[nome] / "tokenizer.json"), "pesos_externos": False,
                             "base_sha256": BASE_SHA[nome], "protocolo": "fase5_v4_" + nome, "commit": COMMIT}},
               DEST / f"pesos_{nome}.pt")
    RESULTADOS[nome]["pesos_sha256"] = sha256(DEST / f"pesos_{nome}.pt")
shutil.copyfile(BASES["F"] / "tokenizer.json", DEST / "tokenizer.json")
linhas = {n: RESULTADOS[n].pop("_linhas") for n in RESULTADOS}
comum = {"protocolo": "fase5_v4", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "gpu": GPU,
         "dados_v4_sha256": M4["dados_sha256"], "dados_v4_sha256_registrado": m4_registrado["dados_sha256"], "sentido_d": SENTIDO_D,
         "partida_d": {"v4": R0_V4["sentido"], "reais": R0_REAIS},
         "hiperparametros": {"passos": PASSOS, "lote_v4": LOTE_V4, "lote_v1": LOTE_V1, "lote_replay": LOTE_REPLAY,
                             "peso_replay": PESO_REPLAY, "lr": LR, "avaliar_a_cada": AVALIAR_A_CADA, "max_horas": MAX_HORAS},
         "escolhida": escolhida, "ativo_no_chat": False, "substitui_d": "só com a catraca e a decisão do criador"}
(DEST / "resultado_fase5_v4.json").write_text(json.dumps({**comum, "variantes": RESULTADOS, "linhas": linhas},
                                                         ensure_ascii=False, default=float), encoding="utf-8")
# Resumo pequeno, sem as linhas: é o que o assistente lê pelo Drive.
(SAIDA / "resumo_v4.json").write_text(json.dumps({**comum, "variantes": RESULTADOS, "pasta": str(DEST)},
                                                 ensure_ascii=False, indent=1, default=float), encoding="utf-8")
print(f"\nSALVO EM: {DEST}  |  resumo: {SAIDA / 'resumo_v4.json'}", flush=True)
for nome, r in RESULTADOS.items():
    print(f"  {nome}: pesos {r['pesos_sha256']} | sentido v3 {r['testes']['v3']['sentido']['certo']} | v4 {r['testes']['v4']['sentido']['certo']} | catraca={r['catraca_passa']}", flush=True)
print("Avise o assistente: ele lê o resumo_v4.json.", flush=True)
